# Hessian eigenvalues along a branch

This notebook visualizes and inspects results computed by `curve_eigenvalues.py`. The script is the sole implementation of scanning, candidate selection, and optional bifurcation seeding.

In [ ]:
import matplotlib.pyplot as plt

from curve_eigenvalues import (
    minimum_in_range,
    save_bifurcation_seed,
    save_scan,
    scan_branch_eigenvalues,
    scan_coordinate,
    seed_bifurcation,
    target_eigenvalues,
)

## Paper scan configuration

The paper's off-branch eigenvalue figure uses the first 537 accepted points of `bprop_negative`, the third-lowest eigenvalue (index 2), and both the original and period-doubled loop representations.

In [ ]:
data_directory = 'hess_results/bprop_negative'
limit = 537
num_eigenvalues = 8
multipliers = (1, 2)
off_branch_index = 2

In [ ]:
scan = scan_branch_eigenvalues(
    data_directory,
    multipliers=multipliers,
    num_eigenvalues=num_eigenvalues,
    limit=limit,
    frequency_cutoff_mode='base',
    use_existing_sidecars=True,
    write_sidecars=True,
)
save_scan(scan, f'{data_directory}/eigenvalue-scan.pt')

## Paper off-branch eigenvalue plot

In [ ]:
theta1 = scan_coordinate(scan, 'theta1')

fig, ax = plt.subplots(figsize=(6, 3.2))
ax.plot(
    theta1, target_eigenvalues(scan, 1, off_branch_index),
    color='C4', label='Original loop',
)
ax.plot(
    theta1, target_eigenvalues(scan, 2, off_branch_index),
    color='darkblue', linestyle=(0, (5, 5)), label='Period-doubled',
)
ax.set_yscale('log')
ax.set_xlabel(r'$\theta_1$')
ax.set_ylabel('Off-branch eigenvalue')
ax.legend(loc='lower right')
fig.tight_layout(pad=0.5);

## Inspect the low spectrum

Plotting all retained modes is useful diagnostically, but is not the figure used in the paper.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), sharey=True)
for ax, multiplier in zip(axes, multipliers):
    ax.plot(theta1, scan.eigenvalues[multiplier][:, 2:])
    ax.set_yscale('log')
    ax.set_xlabel(r'$\theta_1$')
    ax.set_title(f'Period multiplier {multiplier}')
axes[0].set_ylabel('Eigenvalue')
fig.tight_layout();

## Select and inspect a candidate

The automatic range minimum is a convenience for inspection. Record an explicit point and eigenvector index when producing a research branch.

In [ ]:
candidate_index = minimum_in_range(
    scan, multiplier=2, eigenvalue_index=off_branch_index, start=0, stop=len(scan.matches)
)
{
    'index': candidate_index,
    'match': scan.matches[candidate_index],
    'theta1_degrees': theta1[candidate_index].item(),
    'original_eigenvalue': target_eigenvalues(scan, 1)[candidate_index].item(),
    'period_doubled_eigenvalue': target_eigenvalues(scan, 2)[candidate_index].item(),
}

## Optional bifurcation seed

Constructing a seed performs validation but does not save it. Confirm the selected point, eigenvector, orientation, and diagnostics before setting `SAVE_SEED=True`.

In [ ]:
seed_output = 'hess_results/new_bifurcation'
seed = seed_bifurcation(
    scan,
    seed_output,
    source_point_index=candidate_index,
    period_multiplier=2,
    eigenvector_index=off_branch_index,
    step_angle_degrees=0.5,
    first_step_multiplier=10,
    orientation_component=2,
    orientation_sign=1,
    stationary_starts=(False, False),
)
{
    'source_match': seed.source_match,
    'eigenvalue': seed.eigenvalue,
    'seed_loss': seed.seed_loss,
    'closure_error_degrees': seed.closure_error_degrees,
}

In [ ]:
SAVE_SEED = False
if SAVE_SEED:
    save_bifurcation_seed(seed, overwrite=False)